# Týmový úkol na obraz

## 1) Importy

In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
import datetime

## 2) Načtení datasetu

Rozdělení: train[:40000] / train[40000:50000] / test -> 40k : 10k : 10k

In [ ]:
%pip install importlib_resources

(ds_train, ds_val, ds_test), ds_info = tfds.load(
    'cifar10',
    split=['train[:40000]', 'train[40000:50000]', 'test'],
    as_supervised=True,
    with_info=True,
)

NUM_CLASSES = 10
IMG_SIZE = 32
BATCH_SIZE = 128
EPOCHS = 80

ModuleNotFoundError: No module named 'importlib_resources'

## 3) Předzpracování a augmentace dat

- **Normalizace** pixelů do rozsahu [0, 1]
- **Augmentace** (pouze train): horizontální flip + random crop (padding 4 px) → regularizace proti přetrénování

In [ ]:
def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.pad_to_bounding_box(image, 4, 4, IMG_SIZE + 8, IMG_SIZE + 8)
    image = tf.image.random_crop(image, [IMG_SIZE, IMG_SIZE, 3])
    return image, label

ds_train = (
    ds_train
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(40000)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

ds_val = (
    ds_val
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

ds_test = (
    ds_test
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

## 4) Architektura CNN

Vlastní model (žádný předtrénovaný backbone).  
3 konvoluční bloky s rostoucím počtem filtrů (64 → 128 → 256), každý obsahuje:
- 2× Conv2D (3×3, same) + BatchNormalization + ReLU
- MaxPooling2D (2×2)
- Dropout (0.25)

Klasifikační hlava: Dense 512 + BN + ReLU + Dropout 0.5 + Softmax 10

**Opatření proti přetrénování:** Data augmentace, BatchNorm, Dropout, ReduceLROnPlateau, EarlyStopping.

In [ ]:
model = keras.Sequential([
    # --- Blok 1 ---
    layers.Conv2D(64, (3, 3), padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # --- Blok 2 ---
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # --- Blok 3 ---
    layers.Conv2D(256, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(256, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # --- Klasifikační hlava ---
    layers.Flatten(),
    layers.Dense(512),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax'),
])

model.summary()

## 5) Kompilace modelu

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

## 6) Callbacky

- **ReduceLROnPlateau** – sníží learning rate, pokud se val_loss nezlepšuje (factor 0.5, patience 5)
- **EarlyStopping** – zastaví trénink po 15 epochách bez zlepšení; obnoví nejlepší váhy
- **TensorBoard** – loguje pouze accuracy/loss (bez histogramů)

In [ ]:
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_cb = keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=0,
    write_graph=False,
    write_images=False,
    update_freq='epoch',
)

print(f"TensorBoard log dir: {log_dir}")

## 7) Trénování

In [ ]:
history = model.fit(
    ds_train,
    epochs=EPOCHS,
    validation_data=ds_val,
    callbacks=[lr_scheduler, early_stopping, tensorboard_cb],
)

## 8) Vyhodnocení na testovacích datech

In [ ]:
test_loss, test_acc = model.evaluate(ds_test)
print(f"\nTest accuracy: {test_acc:.4f}")
print(f"Test loss:     {test_loss:.4f}")
print(f"\nTensorBoard logy: {log_dir}")
print("Spusťte: tensorboard --logdir logs/fit")